# MP11 — Super-Resolution Evaluation Dashboard

Expanded executable lesson. Prerequisites: earlier image-processing and array lessons. Learning goals: derive the calculation, predict edge cases, implement it, and interpret controlled real-image evidence. Learner mastery is not asserted.

## Setup and conventions
From repository root run `python -m pip install -e ".[dev]"`. Run `python scripts/metric_experiments.py` after downloading the pinned training scene with the portable quickstart. The notebook reads the committed numerical record; it does not silently download imagery. Arrays are HWC float64 for metrics.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
root=Path.cwd()
if not (root/'src').exists(): root=root.parent
sys.path.insert(0,str(root/'src'))
from rssr.metric_lab import mse_psnr,ssim_map


## Mathematics and worked example
A comparison is valid only if predictions and reference share grids, bands, units, masks and crop policy. This static dashboard fixes each of these before measuring. Its first row displays the red band with a common range determined from the reference crop, the second row displays mean absolute error over four bands with a shared zero-to-maximum scale, and the third row displays mean band SSIM on valid windows with a common [-1,1] scale. Display stretching never changes metric inputs.

Aggregate at the level of the claim: patches within a scene are correlated; a hundred patches are not a hundred independent regions. For per-image MSE .01 and .0001 at L=1, image PSNRs are 20 and 40 dB. Their mean is 30 dB. PSNR of mean MSE is -10 log10(.00505)=22.9671 dB. These answer different questions. State which you report, and do not average missing-reference images into a fabricated score.

The dashboard is a controlled-distortion exercise on a training crop, not a leaderboard. Identical, blur, contrast and seeded noise cases expose numerical and display errors. Matched-MSE noise shows why one scalar is insufficient; band RMSE prevents an RGB display from concealing NIR errors.

In [ ]:
per_image=np.array([.01,.0001])
mean_psnr=np.mean(-10*np.log10(per_image))
pooled_psnr=-10*np.log10(per_image.mean())
assert mean_psnr==30 and np.isclose(pooled_psnr,22.96708622)
print(mean_psnr,pooled_psnr)

## Recorded real-data experiment
The source checksum, acquisition date, license, normalization, seed and settings are in the record below. The input is a 64×64 training crop of real four-band NAIP aerial imagery; these are synthetic distortions, not satellite sensor measurements. Reproduce with `python scripts/metric_experiments.py`.

In [ ]:
record=json.loads((root/'reports/mp09-mp11-experiments.json').read_text())
print(record['source']['id'],record['source']['sha256'])
print(record['protocol'])

## Practice
Predict before running: (1) Why use a shared error color scale? (2) Should an unreferenced Sentinel estimate show PSNR? (3) What is the sampling unit for a geographic uncertainty claim?

## Worked solutions
(1) Independently stretched errors can make a poor result look equally good. (2) No, mark it unavailable and report provenance. (3) Independent scenes/regions, with repeated training seeds reported separately; this one crop cannot establish geographic uncertainty.

In [ ]:
assert record['protocol']['clipping'] is False
for row in record['rows']: print(row['case'],row['psnr_db'],row['ssim'])

## Figure and interpretation
See [dashboard](../reports/mp09-mp11-dashboard.png). Blur and matched-MSE noise have equal PSNR but different SSIM. All error maps share a scale. Contrast and blur affect this low-texture crop less than the .01 noise case; this observation is not a universal ranking. Reflect: explain a failure mode that both metrics could miss.

## References and limitations
[Official scikit-image metric documentation](https://scikit-image.org/docs/stable/api/skimage.metrics.html), checked 2026-10-19, documents explicit ranges and SSIM settings. Our NumPy implementation uses declared uniform population moments and valid windows; it does not claim default-package equivalence. No alignment correction, learned perceptual metric or independent-scene claim is made.